# LeetCode 4100 — Minimum Cost Path With Even XOR Parity (Hard)

Bài toán LeetCode:

```python
class Solution:
    def minActivationCost(self, edges: List[List[int]], values: List[int]) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4100 (Weekly Contest 526 — câu 4, rating khoảng 2281).

Bài này trông như một bài "đường đi trên cây với điều kiện XOR" phức tạp, nhưng hai quan sát biến nó thành một bài rất gọn:

1. **Tính chẵn lẻ của số bit 1 "đi xuyên" qua phép XOR**: `popcount(a ^ b)` chẵn/lẻ chỉ phụ thuộc vào `popcount(a)` và `popcount(b)` chẵn/lẻ.
2. **Đường đi rẻ nhất luôn rất ngắn**: không quá 2 cạnh!

Trên đường đi tới đó, ta cũng luyện một kỹ thuật DP trên cây với trạng thái chẵn/lẻ.

---

## Sau notebook này, bạn sẽ nắm được

1. `popcount`, XOR, và tính chất `parity(a ^ b) = parity(a) ^ parity(b)`.
2. Brute Force `O(n²)`: duyệt từ mọi đỉnh.
3. Tree DP với hai trạng thái chẵn/lẻ: `O(n)`.
4. Lập luận "đường đi tối ưu có không quá 2 cạnh" và lời giải một lượt qua các cạnh.
5. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho một **cây** `n` đỉnh (`0..n-1`), mảng cạnh `edges[i] = [u, v, c]` (cạnh vô hướng giữa `u`, `v`, chi phí `c`), và mảng `values` (giá trị của từng đỉnh).

Một đường đi giữa **hai đỉnh khác nhau** được gọi là **cân bằng (Balanced-XOR)** nếu XOR của giá trị **mọi đỉnh trên đường đi** (tính cả hai đầu) có **số bit 1 chẵn**. (Số 0 có 0 bit 1 → chẵn.)

Chi phí của đường đi = tổng chi phí các cạnh. Trả về chi phí nhỏ nhất của một đường đi cân bằng, hoặc `-1` nếu không có.

---

## Ví dụ 1

```python
edges = [[0,1,4],[0,2,6],[0,3,3],[3,4,5]], values = [3,1,7,1,3]   ->   7
```

```text
        1 (1)
        | 4
  2 (7)—6—0 (3)—3—3 (1)—5—4 (3)
```

| Đường đi | XOR | số bit 1 | chi phí |
|---|---|---:|---:|
| 1 – 0 – 2 | `1^3^7 = 5 (101)` | 2 ✔ | 10 |
| 1 – 0 – 3 | `1^3^1 = 3 (011)` | 2 ✔ | **7** |
| 1 – 0 – 3 – 4 | `1^3^1^3 = 0` | 0 ✔ | 12 |
| 2 – 0 – 3 | `7^3^1 = 5` | 2 ✔ | 9 |
| 0 – 1 | `3^1 = 2 (10)` | 1 ✗ | 4 |

Đáp án `7`.

## Ví dụ 2

```python
edges = [[0,1,5],[1,2,10]], values = [1,7,3]   ->   5
```

`0 – 1`: `1 ^ 7 = 6 (110)` có 2 bit → cân bằng, chi phí 5.

## Ví dụ 3

```python
edges = [[0,1,5],[0,2,7],[0,3,9]], values = [1,3,5,9]   ->   -1
```

Mọi đường đi đều có XOR với số bit 1 lẻ.

---

## Ràng buộc

```text
2 <= n <= 10^5,  edges.length == n - 1
1 <= c <= 10^9,  1 <= values[i] <= 10^9
edges tạo thành một cây hợp lệ
```

# 2. Kiến thức nền tảng

## 2.1. XOR và `popcount`

- `a ^ b`: bit kết quả bằng 1 khi hai bit **khác nhau**.
- `popcount(x)`: số bit 1 của `x`. Trong Python: `bin(x).count("1")` hoặc `x.bit_count()` (Python 3.10+).

In [1]:
def popcount(x: int) -> int:
    return bin(x).count("1")


for x in [0, 1, 3, 5, 6, 7]:
    print(f"{x} = {x:03b} -> popcount = {popcount(x)}")
print("3 ^ 1 =", 3 ^ 1, "| 1 ^ 3 ^ 7 =", 1 ^ 3 ^ 7)

0 = 000 -> popcount = 0
1 = 001 -> popcount = 1
3 = 011 -> popcount = 2
5 = 101 -> popcount = 2
6 = 110 -> popcount = 2
7 = 111 -> popcount = 3
3 ^ 1 = 2 | 1 ^ 3 ^ 7 = 5


## 2.2. Tính chất then chốt: chẵn lẻ của `popcount` "đi xuyên" qua XOR

Gọi `parity(x) = popcount(x) % 2`. Khi đó:

```text
parity(a ^ b) = parity(a) ^ parity(b)
```

**Vì sao?** Gọi `k` là số vị trí bit mà **cả** `a` và `b` đều bằng 1. Ở những vị trí đó, XOR cho 0. Vậy:

```text
popcount(a ^ b) = popcount(a) + popcount(b) - 2k
```

Trừ đi một số chẵn `2k` không làm đổi tính chẵn lẻ. ∎

Mở rộng cho nhiều số: `parity(v1 ^ v2 ^ ... ^ vt) = parity(v1) ^ parity(v2) ^ ... ^ parity(vt)`.

Hệ quả: **XOR của đường đi có số bit 1 chẵn ⇔ số đỉnh "lẻ" (đỉnh có `popcount` lẻ) trên đường đi là số chẵn.** Giá trị cụ thể không còn quan trọng — mỗi đỉnh chỉ còn là một bit `0` (chẵn) hoặc `1` (lẻ).

In [2]:
import random

parity = lambda x: popcount(x) & 1
random.seed(0)
for _ in range(10000):
    a, b = random.randint(0, 10**9), random.randint(0, 10**9)
    assert parity(a ^ b) == parity(a) ^ parity(b)
print("parity(a ^ b) == parity(a) ^ parity(b) đúng trên 10000 cặp ngẫu nhiên")

values = [3, 1, 7, 1, 3]
print("values  =", values)
print("parity  =", [parity(v) for v in values], "(1 = đỉnh lẻ)")

parity(a ^ b) == parity(a) ^ parity(b) đúng trên 10000 cặp ngẫu nhiên
values  = [3, 1, 7, 1, 3]
parity  = [0, 1, 1, 1, 0] (1 = đỉnh lẻ)


Với ví dụ 1: đỉnh `0` và `4` (giá trị 3) là **chẵn**; đỉnh `1, 2, 3` là **lẻ**. Đường `1 – 0 – 3` có 2 đỉnh lẻ (1 và 3) → cân bằng.

## 2.3. Danh sách kề có trọng số và duyệt cây bằng stack

Cạnh vô hướng → thêm cả hai chiều `(đỉnh kề, chi phí)`. Duyệt bằng stack (không đệ quy) để an toàn với cây sâu `10^5`.

In [3]:
edges = [[0, 1, 4], [0, 2, 6], [0, 3, 3], [3, 4, 5]]
n = 5
adj = [[] for _ in range(n)]
for u, v, c in edges:
    adj[u].append((v, c))
    adj[v].append((u, c))
for u in range(n):
    print(u, "->", adj[u])

0 -> [(1, 4), (2, 6), (3, 3)]
1 -> [(0, 4)]
2 -> [(0, 6)]
3 -> [(0, 3), (4, 5)]
4 -> [(3, 5)]


# 3. Cách 1 — Brute Force: từ mỗi đỉnh, đi tới mọi đỉnh khác

Từ mỗi đỉnh nguồn `s`, duyệt cây, mang theo **XOR tích lũy** và **chi phí tích lũy**. Mỗi đỉnh tới được (khác `s`) là một đường đi; kiểm tra điều kiện và cập nhật đáp án.

In [4]:
from typing import List


def min_cost_bruteforce(edges: List[List[int]], values: List[int]) -> int:
    n = len(values)
    adj = [[] for _ in range(n)]
    for u, v, c in edges:
        adj[u].append((v, c))
        adj[v].append((u, c))
    best = float("inf")
    for s in range(n):
        stack = [(s, -1, values[s], 0)]           # (đỉnh, cha, XOR tích lũy, chi phí)
        while stack:
            u, par, x, cost = stack.pop()
            if u != s and popcount(x) % 2 == 0:
                best = min(best, cost)
            for v, c in adj[u]:
                if v != par:
                    stack.append((v, u, x ^ values[v], cost + c))
    return -1 if best == float("inf") else best


print(min_cost_bruteforce([[0, 1, 4], [0, 2, 6], [0, 3, 3], [3, 4, 5]], [3, 1, 7, 1, 3]))  # 7
print(min_cost_bruteforce([[0, 1, 5], [1, 2, 10]], [1, 7, 3]))                         # 5
print(min_cost_bruteforce([[0, 1, 5], [0, 2, 7], [0, 3, 9]], [1, 3, 5, 9]))            # -1

7
5
-1


`n` lần duyệt, mỗi lần `O(n)` → `O(n²)`. Với `n = 10^5` là `10^10` — quá chậm.

---

# 4. Cách 2 — Tree DP với hai trạng thái chẵn/lẻ: O(n)

Nhờ mục 2.2, mỗi đỉnh chỉ còn một bit `par[u]`. Chọn gốc `0`. Với mỗi đỉnh `u`, định nghĩa:

```text
dp[u][b] = chi phí nhỏ nhất của một đường đi "đi xuống" bắt đầu tại u (u là đầu trên),
           kết thúc ở một đỉnh trong cây con của u, có tổng chẵn lẻ (số đỉnh lẻ mod 2) bằng b
```

(Đường chỉ có mình `u` cũng tính: `dp[u][par[u]] = 0`.)

Xử lý các con `v` của `u` lần lượt (con trước cha). Với cạnh `(u, v, c)`:

1. **Ghép thành đường đi đi qua `u`**: một nhánh đã có trong `dp[u]` (chẵn lẻ `b1`) và một nhánh mới đi xuống qua `v` (chẵn lẻ `b2`, chi phí `dp[v][b2] + c`). Đỉnh `u` đã tính trong nhánh `dp[u]`, nhánh qua `v` không chứa `u`. Tổng chẵn lẻ là `b1 ^ b2`; cân bằng ⇔ `b1 == b2`:

   ```text
   answer = min(answer, dp[u][b] + c + dp[v][b])     với b = 0, 1
   ```

   Vì nhánh qua `v` có ít nhất đỉnh `v`, đường ghép luôn có ít nhất 2 đỉnh.

2. **Cập nhật `dp[u]`** bằng các đường đi xuống qua `v`: thêm đỉnh `u` vào đầu đổi chẵn lẻ theo `par[u]`:

   ```text
   dp[u][b ^ par[u]] = min(dp[u][b ^ par[u]], dp[v][b] + c)
   ```

Thứ tự (ghép trước, cập nhật sau) đảm bảo không ghép một nhánh với chính nó.

In [5]:
def min_cost_tree_dp(edges: List[List[int]], values: List[int]) -> int:
    n = len(values)
    par = [popcount(v) & 1 for v in values]
    adj = [[] for _ in range(n)]
    for u, v, c in edges:
        adj[u].append((v, c))
        adj[v].append((u, c))

    parent = [-1] * n
    parent_cost = [0] * n
    order = []
    seen = [False] * n
    seen[0] = True
    stack = [0]
    while stack:
        u = stack.pop()
        order.append(u)
        for v, c in adj[u]:
            if not seen[v]:
                seen[v] = True
                parent[v] = u
                parent_cost[v] = c
                stack.append(v)

    INF = float("inf")
    dp = [[INF, INF] for _ in range(n)]
    for u in range(n):
        dp[u][par[u]] = 0
    best = INF
    for v in reversed(order):                    # con trước cha
        u = parent[v]
        if u == -1:
            continue
        c = parent_cost[v]
        for b in (0, 1):                         # 1. ghép đường đi qua u
            best = min(best, dp[u][b] + c + dp[v][b])
        for b in (0, 1):                         # 2. cập nhật dp[u]
            dp[u][b ^ par[u]] = min(dp[u][b ^ par[u]], dp[v][b] + c)
    return -1 if best == INF else best


print(min_cost_tree_dp([[0, 1, 4], [0, 2, 6], [0, 3, 3], [3, 4, 5]], [3, 1, 7, 1, 3]))  # 7
print(min_cost_tree_dp([[0, 1, 5], [1, 2, 10]], [1, 7, 3]))                         # 5
print(min_cost_tree_dp([[0, 1, 5], [0, 2, 7], [0, 3, 9]], [1, 3, 5, 9]))            # -1

7
5
-1


Cách 2 đã `O(n)` và đúng. Nhưng hãy dừng lại và hỏi: **đường đi tối ưu trông như thế nào?** Câu trả lời dẫn tới một lời giải còn đơn giản hơn nhiều.

---

# 5. Bước đột phá: đường đi tối ưu có không quá 2 cạnh

Mọi chi phí cạnh đều **dương** (`c >= 1`). Vậy nếu một đường đi cân bằng chứa bên trong nó một **đường đi con** cũng cân bằng (ngắn hơn), thì đường đi con **rẻ hơn hẳn**.

Xét một đường đi cân bằng bất kỳ với **ít nhất 3 cạnh** (ít nhất 4 đỉnh). Nhìn dãy bit chẵn/lẻ của các đỉnh dọc đường:

- **Nếu có hai đỉnh kề nhau cùng bit** (`0-0` hoặc `1-1`): riêng cạnh đó là một đường cân bằng (số đỉnh lẻ là 0 hoặc 2) — rẻ hơn.
- **Nếu không** thì bit xen kẽ: `0 1 0 1 ...` hoặc `1 0 1 0 ...`. Với ít nhất 4 đỉnh, luôn có một bộ ba liên tiếp dạng `1 0 1` — đường 2 cạnh này có đúng 2 đỉnh lẻ → cân bằng và rẻ hơn.

Vậy **đường đi cân bằng rẻ nhất luôn là một trong hai dạng**:

| Dạng | Mô tả | Số đỉnh lẻ |
|---|---|---:|
| **A** — 1 cạnh | hai đầu **cùng** bit (`0-0` hoặc `1-1`) | 0 hoặc 2 ✔ |
| **B** — 2 cạnh | `lẻ – chẵn – lẻ` (đỉnh giữa chẵn, hai đầu lẻ) | 2 ✔ |

(Đường 2 cạnh khác như `0-1-0` có 1 đỉnh lẻ ✗; `0-0-x` hay `1-1-x` thì chứa một cạnh dạng A rẻ hơn.)

## Tính dạng B nhanh

Với mỗi đỉnh **chẵn** `u`, ta cần **hai cạnh rẻ nhất** từ `u` tới các hàng xóm **lẻ**. Duyệt các cạnh một lượt, giữ `mn[u]` = cạnh rẻ nhất đã thấy từ `u` tới hàng xóm lẻ. Khi gặp cạnh mới `(u, v, c)` với `u` chẵn, `v` lẻ: ghép `mn[u] + c` thành ứng viên, rồi cập nhật `mn[u]`.

# 6. Cách 3 — Một lượt qua các cạnh: O(n)

In [6]:
def min_cost_two_edges(edges: List[List[int]], values: List[int]) -> int:
    par = [popcount(v) & 1 for v in values]
    INF = float("inf")
    best = INF
    mn = [INF] * len(values)          # mn[u]: cạnh rẻ nhất từ đỉnh chẵn u tới hàng xóm lẻ
    for u, v, c in edges:
        if par[u] == par[v]:          # dạng A
            best = min(best, c)
            continue
        if par[u] == 1:               # đưa đỉnh chẵn về u
            u, v = v, u
        best = min(best, mn[u] + c)   # dạng B: ghép với cạnh rẻ nhất trước đó tại u
        mn[u] = min(mn[u], c)         # cập nhật SAU khi ghép
    return -1 if best == INF else best


print(min_cost_two_edges([[0, 1, 4], [0, 2, 6], [0, 3, 3], [3, 4, 5]], [3, 1, 7, 1, 3]))  # 7
print(min_cost_two_edges([[0, 1, 5], [1, 2, 10]], [1, 7, 3]))                         # 5
print(min_cost_two_edges([[0, 1, 5], [0, 2, 7], [0, 3, 9]], [1, 3, 5, 9]))            # -1

7
5
-1


## Mô phỏng ví dụ 1

Bit chẵn/lẻ: `par = [0, 1, 1, 1, 0]`.

| Cạnh (u, v, c) | bit | dạng | ứng viên | best | mn sau |
|---|---|---|---:|---:|---|
| (0, 1, 4) | 0 – 1 | B, tâm 0 | `mn[0] + 4 = ∞` | ∞ | `mn[0] = 4` |
| (0, 2, 6) | 0 – 1 | B, tâm 0 | `4 + 6 = 10` | 10 | `mn[0] = 4` |
| (0, 3, 3) | 0 – 1 | B, tâm 0 | `4 + 3 = 7` | 7 | `mn[0] = 3` |
| (3, 4, 5) | 1 – 0 | B, tâm 4 | `mn[4] + 5 = ∞` | 7 | `mn[4] = 5` |

Đáp án `7` — đường `1 – 0 – 3` (hai cạnh rẻ nhất 4 và 3 quanh tâm chẵn `0`).

In [7]:
def trace_two_edges(edges: List[List[int]], values: List[int]) -> int:
    par = [popcount(v) & 1 for v in values]
    print("par =", par)
    INF = float("inf")
    best = INF
    mn = [INF] * len(values)
    for u, v, c in edges:
        if par[u] == par[v]:
            best = min(best, c)
            print(f"cạnh ({u},{v},{c}): cùng bit {par[u]} -> dạng A, ứng viên {c}, best = {best}")
            continue
        if par[u] == 1:
            u, v = v, u
        cand = mn[u] + c
        best = min(best, cand)
        mn[u] = min(mn[u], c)
        print(f"cạnh ({u},{v},{c}): tâm chẵn {u} -> dạng B, ứng viên {cand}, best = {best}, mn[{u}] = {mn[u]}")
    ans = -1 if best == INF else best
    print("Đáp án:", ans)
    return ans


trace_two_edges([[0, 1, 4], [0, 2, 6], [0, 3, 3], [3, 4, 5]], [3, 1, 7, 1, 3]);

par = [0, 1, 1, 1, 0]
cạnh (0,1,4): tâm chẵn 0 -> dạng B, ứng viên inf, best = inf, mn[0] = 4
cạnh (0,2,6): tâm chẵn 0 -> dạng B, ứng viên 10, best = 10, mn[0] = 4
cạnh (0,3,3): tâm chẵn 0 -> dạng B, ứng viên 7, best = 7, mn[0] = 3
cạnh (4,3,5): tâm chẵn 4 -> dạng B, ứng viên inf, best = 7, mn[4] = 5
Đáp án: 7


# 7. Phiên bản đúng format LeetCode

In [8]:
from typing import List


class Solution:
    def minActivationCost(self, edges: List[List[int]], values: List[int]) -> int:
        par = [bin(v).count("1") & 1 for v in values]
        INF = float("inf")
        best = INF
        mn = [INF] * len(values)
        for u, v, c in edges:
            if par[u] == par[v]:
                if c < best:
                    best = c
                continue
            if par[u] == 1:
                u, v = v, u
            if mn[u] + c < best:
                best = mn[u] + c
            if c < mn[u]:
                mn[u] = c
        return -1 if best == INF else best

In [9]:
sol = Solution()
tests = [
    ([[0, 1, 4], [0, 2, 6], [0, 3, 3], [3, 4, 5]], [3, 1, 7, 1, 3], 7, "ví dụ 1"),
    ([[0, 1, 5], [1, 2, 10]], [1, 7, 3], 5, "ví dụ 2"),
    ([[0, 1, 5], [0, 2, 7], [0, 3, 9]], [1, 3, 5, 9], -1, "ví dụ 3"),
    ([[0, 1, 9]], [3, 5], 9, "n = 2, hai đỉnh cùng chẵn"),
    ([[0, 1, 9]], [1, 3], -1, "n = 2, khác bit"),
    ([[0, 1, 2], [1, 2, 1], [1, 3, 5], [3, 4, 1]], [1, 3, 2, 6, 4], 3, "bài tập 1"),
]
for e, v, expected, name in tests:
    result = sol.minActivationCost(e, v)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:28} -> {result}")

PASS | ví dụ 1                      -> 7
PASS | ví dụ 2                      -> 5
PASS | ví dụ 3                      -> -1
PASS | n = 2, hai đỉnh cùng chẵn    -> 9
PASS | n = 2, khác bit              -> -1
PASS | bài tập 1                    -> 3


# 8. Vì sao đúng?

1. **Rút gọn điều kiện** (mục 2.2): đường cân bằng ⇔ số đỉnh lẻ chẵn.
2. **Đường tối ưu ≤ 2 cạnh** (mục 5): mọi đường cân bằng ≥ 3 cạnh chứa một đường con cân bằng ngắn hơn, mà chi phí cạnh dương → không thể tối ưu.
3. **Liệt kê đủ hai dạng**: dạng A là mọi cạnh cùng bit; dạng B là mọi cặp cạnh `(u, v1)`, `(u, v2)` với `u` chẵn, `v1 ≠ v2` lẻ. Với mỗi tâm `u`, cặp rẻ nhất là hai cạnh rẻ nhất; cách duyệt "ghép với `mn[u]` rồi mới cập nhật" xét đúng cặp đó (khi cạnh rẻ nhì xuất hiện sau cạnh rẻ nhất hoặc ngược lại, ở thời điểm cạnh xuất hiện sau, `mn[u]` đã chứa cạnh kia).
4. Không có đường nào → `-1`.

---

## Trực giác

> Đường càng dài càng tốn tiền, mà cứ dài từ 3 cạnh trở lên thì bên trong luôn có một "đoạn ngắn cũng đạt yêu cầu". Vậy chỉ cần soi các đoạn rất ngắn.

# 9. Độ phức tạp

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---:|---:|
| 1. Brute Force | duyệt từ mọi đỉnh | `O(n²)` | `O(n)` |
| 2. Tree DP | `dp[u][chẵn/lẻ]` | `O(n)` | `O(n)` |
| 3. Hai cạnh | một lượt qua cạnh | `O(n)` | `O(n)` |

(Tính `popcount` của số `<= 10^9` tốn `O(log V)` — coi như hằng số.)

# 10. Đối chiếu trên cây ngẫu nhiên

In [10]:
random.seed(4100)
for _ in range(3000):
    n_ = random.randint(2, 9)
    e_ = []
    for i in range(1, n_):
        p = random.randrange(i)
        e_.append([p, i, random.randint(1, 10)] if random.random() < 0.5 else [i, p, random.randint(1, 10)])
    v_ = [random.randint(1, 20) for _ in range(n_)]
    expected = min_cost_bruteforce(e_, v_)
    assert min_cost_tree_dp(e_, v_) == expected, (e_, v_)
    assert min_cost_two_edges(e_, v_) == expected, (e_, v_)
    assert sol.minActivationCost(e_, v_) == expected, (e_, v_)
print("3000 cây ngẫu nhiên: 4 hàm cho cùng kết quả!")

3000 cây ngẫu nhiên: 4 hàm cho cùng kết quả!


In [11]:
import time

N = 10**5
big_edges = [[random.randrange(i), i, random.randint(1, 10**9)] for i in range(1, N)]
big_values = [random.randint(1, 10**9) for _ in range(N)]
t0 = time.perf_counter()
r3 = sol.minActivationCost(big_edges, big_values)
t3 = time.perf_counter() - t0
t0 = time.perf_counter()
r2 = min_cost_tree_dp(big_edges, big_values)
t2 = time.perf_counter() - t0
print(f"n = 10^5: hai cạnh = {r3} ({t3:.2f}s) | tree DP = {r2} ({t2:.2f}s)")

n = 10^5: hai cạnh = 30167 (0.04s) | tree DP = 30167 (0.37s)


# 11. Những lỗi dễ mắc

## Lỗi 1 — Chỉ XOR hai đầu mút

Đề yêu cầu XOR của **mọi đỉnh** trên đường đi. Nếu chỉ XOR hai đầu, đường `1 – 0 – 3` ở ví dụ 1 có `1 ^ 1 = 0` (cân bằng) — đúng tình cờ, nhưng nhiều đường khác sai. Hãy luôn đối chiếu bằng brute force.

## Lỗi 2 — Ghép dạng B với tâm **lẻ**

Đường `chẵn – lẻ – chẵn` chỉ có **1** đỉnh lẻ → **không** cân bằng.

In [12]:
def wrong_odd_center(edges, values):
    par = [popcount(v) & 1 for v in values]
    INF = float("inf")
    best = INF
    mn = [INF] * len(values)
    for u, v, c in edges:
        if par[u] == par[v]:
            best = min(best, c)
            continue
        for w in (u, v):                      # SAI: ghép ở cả hai đầu, kể cả tâm lẻ
            best = min(best, mn[w] + c)
            mn[w] = min(mn[w], c)
    return -1 if best == INF else best


print("Sai :", wrong_odd_center([[0, 1, 5], [0, 2, 7], [0, 3, 9]], [1, 3, 5, 9]))
print("Đúng:", sol.minActivationCost([[0, 1, 5], [0, 2, 7], [0, 3, 9]], [1, 3, 5, 9]))

Sai : 12
Đúng: -1


Ở ví dụ 3, tâm `0` là đỉnh **lẻ** (giá trị 1), ba lá đều chẵn. Code sai ghép hai cạnh quanh tâm lẻ → đường `chẵn – lẻ – chẵn` (1 đỉnh lẻ) bị coi là hợp lệ.

## Lỗi 3 — Cập nhật `mn[u]` trước khi ghép

Nếu cập nhật trước, cạnh mới sẽ ghép với **chính nó**: đường `v – u – v` (đi rồi quay lại) — không phải đường đi giữa hai đỉnh khác nhau.

In [13]:
def wrong_update_first(edges, values):
    par = [popcount(v) & 1 for v in values]
    INF = float("inf")
    best = INF
    mn = [INF] * len(values)
    for u, v, c in edges:
        if par[u] == par[v]:
            best = min(best, c)
            continue
        if par[u] == 1:
            u, v = v, u
        mn[u] = min(mn[u], c)                  # SAI: cập nhật trước
        best = min(best, mn[u] + c)
    return -1 if best == INF else best


print("Sai :", wrong_update_first([[0, 1, 9]], [3, 1]))
print("Đúng:", sol.minActivationCost([[0, 1, 9]], [3, 1]))

Sai : 18
Đúng: -1


## Lỗi 4 — Cho phép đường đi một đỉnh

Đường đi phải giữa **hai đỉnh khác nhau**. Nếu khởi tạo `best = 0` (đường chỉ một đỉnh chẵn), đáp án luôn bằng 0.

## Lỗi 5 — Đệ quy khi làm tree DP

Cây có thể là đường thẳng `10^5` đỉnh → `RecursionError`. Dùng stack như cách 2.

# 12. Mẹo nhận diện

1. Điều kiện về **chẵn lẻ của `popcount`** của một **XOR** → thay mỗi số bằng một bit `parity`; XOR của các bit.
2. **Chi phí dương** + điều kiện "có đường con thỏa" → đường tối ưu **ngắn** — thử chứng minh "mọi đường dài đều chứa đường con thỏa điều kiện".
3. Đường đi trên cây với trạng thái nhỏ (vài giá trị) → **tree DP** `dp[u][trạng thái]`, ghép tại đỉnh cao nhất.
4. "Hai cạnh rẻ nhất quanh một tâm" → duyệt cạnh, giữ cạnh rẻ nhất đã thấy tại mỗi đỉnh, ghép rồi mới cập nhật.

Các bài liên quan: *Count Paths That Can Form a Palindrome in a Tree* (LeetCode 2791 — cũng dùng XOR/parity trên đường đi), *Binary Tree Maximum Path Sum* (124 — ghép hai nhánh tại đỉnh cao nhất).

# 13. Biến thể

## Biến thể 1 — Đường đi cân bằng ít cạnh nhất

Theo mục 5: có cạnh cùng bit → `1`; có đỉnh chẵn với ít nhất 2 hàng xóm lẻ → `2`; ngược lại `-1`.

In [14]:
def fewest_edges(edges: List[List[int]], values: List[int]) -> int:
    par = [popcount(v) & 1 for v in values]
    odd_neighbors = [0] * len(values)
    for u, v, _ in edges:
        if par[u] == par[v]:
            return 1
        even = u if par[u] == 0 else v
        odd_neighbors[even] += 1
    return 2 if max(odd_neighbors) >= 2 else -1


print(fewest_edges([[0, 1, 4], [0, 2, 6], [0, 3, 3], [3, 4, 5]], [3, 1, 7, 1, 3]))   # 2
print(fewest_edges([[0, 1, 5], [1, 2, 10]], [1, 7, 3]))                           # 1

2
1


## Biến thể 2 — Tree DP dùng được cho cả chi phí âm

Nếu cạnh được phép có chi phí **âm**, lập luận "đường ngắn luôn rẻ hơn" sụp đổ, cách 3 sai — nhưng tree DP (cách 2) vẫn đúng vì nó xét **mọi** đường đi. Đây là lý do nên biết cả hai cách.

In [15]:
neg_edges = [[0, 1, -5], [1, 2, -5], [2, 3, -5]]
neg_values = [3, 1, 1, 3]        # bit: 0 1 1 0
print("brute force:", min_cost_bruteforce(neg_edges, neg_values))
print("tree DP    :", min_cost_tree_dp(neg_edges, neg_values))
print("hai cạnh   :", min_cost_two_edges(neg_edges, neg_values), "(sai khi chi phí âm)")

brute force: -15
tree DP    : -15
hai cạnh   : -5 (sai khi chi phí âm)


# 14. Template tư duy

```python
par = [popcount(v) & 1 for v in values]      # 1. rút gọn về bit chẵn/lẻ
best = INF; mn = [INF] * n
for u, v, c in edges:                         # 2. dạng A và B
    if par[u] == par[v]:
        best = min(best, c); continue
    if par[u] == 1: u, v = v, u               #    u là tâm chẵn
    best = min(best, mn[u] + c)               #    ghép trước
    mn[u] = min(mn[u], c)                     #    cập nhật sau
return -1 if best == INF else best
```

# 15. Bài tập vận dụng

## Bài 1 — Tính bằng tay

```python
edges = [[0,1,2],[1,2,1],[1,3,5],[3,4,1]], values = [1,3,2,6,4]
```

Tính bit chẵn/lẻ của từng đỉnh, rồi mô phỏng cách 3. Kiểm tra bằng cell dưới.

In [16]:
# Kiểm tra Bài 1
trace_two_edges([[0, 1, 2], [1, 2, 1], [1, 3, 5], [3, 4, 1]], [1, 3, 2, 6, 4]);

par = [1, 0, 1, 0, 1]
cạnh (1,0,2): tâm chẵn 1 -> dạng B, ứng viên inf, best = inf, mn[1] = 2
cạnh (1,2,1): tâm chẵn 1 -> dạng B, ứng viên 3, best = 3, mn[1] = 1
cạnh (1,3,5): cùng bit 0 -> dạng A, ứng viên 5, best = 3
cạnh (3,4,1): tâm chẵn 3 -> dạng B, ứng viên inf, best = 3, mn[3] = 1
Đáp án: 3


## Bài 2 — Dự đoán output

```python
print(popcount(10), popcount(10 ^ 7), (popcount(10) + popcount(7)) % 2)
print([popcount(v) & 1 for v in [8, 9, 15, 16]])
```

In [17]:
print(popcount(10), popcount(10 ^ 7), (popcount(10) + popcount(7)) % 2)
print([popcount(v) & 1 for v in [8, 9, 15, 16]])

2 3 1
[1, 0, 0, 1]


## Bài 3 — Hoàn thiện cách "hai cạnh"

In [18]:
def exercise_3_two_edges(edges: List[List[int]], values: List[int]) -> int:
    # TODO 1: par = bit chẵn/lẻ của popcount
    # TODO 2: duyệt cạnh: cùng bit -> ứng viên c; khác bit -> đưa tâm chẵn về u, ghép mn[u] + c, cập nhật mn[u]
    # TODO 3: trả -1 nếu không có ứng viên
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_two_edges([[0, 1, 4], [0, 2, 6], [0, 3, 3], [3, 4, 5]], [3, 1, 7, 1, 3]) == 7

## Bài 4 — Đếm số cặp cạnh dạng B

Đếm số đường đi dạng `lẻ – chẵn – lẻ` (không quan tâm chi phí). Với tâm chẵn có `k` hàng xóm lẻ, có `k·(k-1)/2` đường.

```python
count_type_b([[0,1,4],[0,2,6],[0,3,3],[3,4,5]], [3,1,7,1,3])   -> 3
```

In [19]:
def exercise_4_count_type_b(edges: List[List[int]], values: List[int]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_count_type_b([[0, 1, 4], [0, 2, 6], [0, 3, 3], [3, 4, 5]], [3, 1, 7, 1, 3]) == 3

## Bài 5 — Biến thể: số bit 1 **lẻ**

Đổi điều kiện: đường đi hợp lệ nếu XOR có số bit 1 **lẻ** (tức số đỉnh lẻ là **lẻ**). Tìm chi phí nhỏ nhất (chi phí cạnh dương). Hãy tự lập luận các dạng đường tối ưu, rồi kiểm tra bằng brute force tự viết.

Gợi ý: cạnh **khác bit** là hợp lệ ngay (1 đỉnh lẻ). Nếu mọi cạnh cùng bit thì sao?

In [20]:
def exercise_5_odd_version(edges: List[List[int]], values: List[int]) -> int:
    # TODO
    raise NotImplementedError


# print(exercise_5_odd_version([[0, 1, 4], [0, 2, 6], [0, 3, 3], [3, 4, 5]], [3, 1, 7, 1, 3]))

## Bài 6 — Mini challenge: trả về chính đường đi

Trả về **danh sách đỉnh** của một đường đi cân bằng rẻ nhất (hoặc `[]`).

```python
best_path([[0,1,4],[0,2,6],[0,3,3],[3,4,5]], [3,1,7,1,3])   -> [1, 0, 3]  (hoặc [3, 0, 1])
```

In [21]:
def exercise_6_best_path(edges: List[List[int]], values: List[int]) -> List[int]:
    # TODO
    raise NotImplementedError


# path = exercise_6_best_path([[0, 1, 4], [0, 2, 6], [0, 3, 3], [3, 4, 5]], [3, 1, 7, 1, 3])
# assert sorted(path) == [0, 1, 3]

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: `par = [popcount(v) & 1 for v in values]`.
- Mức 2: nhánh cùng bit: `best = min(best, c)`.
- Mức 3: nhánh khác bit: `if par[u] == 1: u, v = v, u`; `best = min(best, mn[u] + c)`; `mn[u] = min(mn[u], c)`.

**Bài 4**

- Mức 1: đếm số hàng xóm lẻ của mỗi đỉnh chẵn.
- Mức 2: chỉ xét cạnh khác bit.
- Mức 3: cộng `k*(k-1)//2`.

**Bài 5**

- Mức 1: cạnh khác bit → hợp lệ.
- Mức 2: nếu chỉ dùng cạnh cùng bit: đường toàn đỉnh chẵn luôn có 0 đỉnh lẻ ✗; đường toàn đỉnh lẻ với 3 đỉnh (2 cạnh) có 3 đỉnh lẻ ✔.
- Mức 3: ứng viên = mọi cạnh khác bit, và hai cạnh rẻ nhất quanh một **tâm lẻ** nối tới hai hàng xóm **lẻ**.

**Bài 6**

- Mức 1: lưu thêm cạnh đạt `mn[u]` (tức hàng xóm `w`).
- Mức 2: khi một ứng viên dạng A thắng, đường là `[u, v]`; dạng B thắng, đường là `[w, u, v]`.
- Mức 3: cập nhật đường đi cùng lúc với `best`.

# 16. Đáp án tham khảo

## Đáp án Bài 1

`popcount`: `1→1, 3→2, 2→1, 6→2, 4→1` → `par = [1, 0, 1, 0, 1]`.

| Cạnh | bit | dạng | ứng viên | best |
|---|---|---|---:|---:|
| (0, 1, 2) | 1 – 0 | B, tâm 1 | ∞ | ∞ (`mn[1] = 2`) |
| (1, 2, 1) | 0 – 1 | B, tâm 1 | `2 + 1 = 3` | 3 (`mn[1] = 1`) |
| (1, 3, 5) | 0 – 0 | A | 5 | 3 |
| (3, 4, 1) | 0 – 1 | B, tâm 3 | ∞ | 3 (`mn[3] = 1`) |

Đáp án `3`: đường `0 – 1 – 2` (lẻ – chẵn – lẻ).

## Đáp án Bài 2

```text
2 3 1
[1, 0, 0, 1]
```

`10 = 1010` có 2 bit; `10 ^ 7 = 1010 ^ 0111 = 1101 = 13` có 3 bit (lẻ). Đúng như tính chất: `(2 + 3) % 2 = 1` (lẻ). Dòng hai: `8 = 1000` (1 bit), `9 = 1001` (2), `15 = 1111` (4), `16 = 10000` (1).

In [22]:
# Đáp án Bài 3
def answer_3_two_edges(edges: List[List[int]], values: List[int]) -> int:
    par = [popcount(v) & 1 for v in values]
    INF = float("inf")
    best = INF
    mn = [INF] * len(values)
    for u, v, c in edges:
        if par[u] == par[v]:
            best = min(best, c)
            continue
        if par[u] == 1:
            u, v = v, u
        best = min(best, mn[u] + c)
        mn[u] = min(mn[u], c)
    return -1 if best == INF else best


assert answer_3_two_edges([[0, 1, 4], [0, 2, 6], [0, 3, 3], [3, 4, 5]], [3, 1, 7, 1, 3]) == 7
print("Bài 3: PASS")

Bài 3: PASS


In [23]:
# Đáp án Bài 4
def answer_4_count_type_b(edges: List[List[int]], values: List[int]) -> int:
    par = [popcount(v) & 1 for v in values]
    k = [0] * len(values)
    for u, v, _ in edges:
        if par[u] != par[v]:
            k[u if par[u] == 0 else v] += 1
    return sum(x * (x - 1) // 2 for x in k)


assert answer_4_count_type_b([[0, 1, 4], [0, 2, 6], [0, 3, 3], [3, 4, 5]], [3, 1, 7, 1, 3]) == 3
print("Bài 4: PASS")

Bài 4: PASS


In [24]:
# Đáp án Bài 5
def answer_5_odd_version(edges: List[List[int]], values: List[int]) -> int:
    par = [popcount(v) & 1 for v in values]
    INF = float("inf")
    best = INF
    mn = [INF] * len(values)            # cạnh rẻ nhất từ đỉnh lẻ tới hàng xóm lẻ
    for u, v, c in edges:
        if par[u] != par[v]:
            best = min(best, c)         # 1 đỉnh lẻ
        elif par[u] == 1:               # cạnh lẻ - lẻ: ghép quanh tâm lẻ
            best = min(best, mn[u] + c, mn[v] + c)
            mn[u] = min(mn[u], c)
            mn[v] = min(mn[v], c)
    return -1 if best == INF else best


def brute_odd(edges, values):
    n = len(values)
    adj = [[] for _ in range(n)]
    for u, v, c in edges:
        adj[u].append((v, c)); adj[v].append((u, c))
    best = float("inf")
    for s in range(n):
        st = [(s, -1, values[s], 0)]
        while st:
            u, p, x, cost = st.pop()
            if u != s and popcount(x) % 2 == 1:
                best = min(best, cost)
            for v, c in adj[u]:
                if v != p:
                    st.append((v, u, x ^ values[v], cost + c))
    return -1 if best == float("inf") else best


random.seed(5)
for _ in range(2000):
    n_ = random.randint(2, 8)
    e_ = [[random.randrange(i), i, random.randint(1, 10)] for i in range(1, n_)]
    v_ = [random.randint(1, 20) for _ in range(n_)]
    assert answer_5_odd_version(e_, v_) == brute_odd(e_, v_), (e_, v_)
print("Bài 5: PASS (khớp brute force trên 2000 cây)")

Bài 5: PASS (khớp brute force trên 2000 cây)


In [25]:
# Đáp án Bài 6
def answer_6_best_path(edges: List[List[int]], values: List[int]) -> List[int]:
    par = [popcount(v) & 1 for v in values]
    INF = float("inf")
    best, path = INF, []
    mn = [INF] * len(values)
    via = [-1] * len(values)            # hàng xóm lẻ đạt mn[u]
    for u, v, c in edges:
        if par[u] == par[v]:
            if c < best:
                best, path = c, [u, v]
            continue
        if par[u] == 1:
            u, v = v, u
        if mn[u] + c < best:
            best, path = mn[u] + c, [via[u], u, v]
        if c < mn[u]:
            mn[u], via[u] = c, v
    return path


path = answer_6_best_path([[0, 1, 4], [0, 2, 6], [0, 3, 3], [3, 4, 5]], [3, 1, 7, 1, 3])
print(path)
assert sorted(path) == [0, 1, 3]
print("Bài 6: PASS")

[1, 0, 3]
Bài 6: PASS


# 17. Tổng kết

**Minimum Cost Path With Even XOR Parity** = hai quan sát + một vòng lặp:

1. `parity(a ^ b) = parity(a) ^ parity(b)` → mỗi đỉnh chỉ là một bit; đường cân bằng ⇔ số đỉnh lẻ chẵn.
2. Chi phí dương → đường tối ưu ≤ 2 cạnh: **cạnh cùng bit**, hoặc **lẻ – chẵn – lẻ**.
3. Một lượt qua cạnh: ghép với cạnh rẻ nhất đã thấy tại tâm chẵn.

Tree DP hai trạng thái là lời giải tổng quát hơn (đúng cả khi chi phí âm).

## Độ phức tạp

```text
Time:  O(n)
Space: O(n)
```

# 18. Checklist trước khi submit

- [ ] Đã rút gọn mỗi giá trị về `popcount(v) & 1`?
- [ ] Dạng A: cạnh hai đầu **cùng** bit?
- [ ] Dạng B: chỉ ghép quanh **tâm chẵn** với hai hàng xóm **lẻ**?
- [ ] Ghép **trước**, cập nhật `mn` **sau** (tránh ghép cạnh với chính nó)?
- [ ] Không coi đường một đỉnh là hợp lệ?
- [ ] Trả `-1` khi không có ứng viên?